In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

import sys

sys.path.append("..")  # Go back to base directory

from modules.graph import *
from modules.viewer3d import *

In [ ]:
def dh_transform(theta, d, a, alpha):
    """Compute individual DH transformation matrix."""
    ct, st = np.cos(theta), np.sin(theta)
    ca, sa = np.cos(alpha), np.sin(alpha)

    return np.array([
        [ct, -st * ca,  st * sa, a * ct],
        [st,  ct * ca, -ct * sa, a * st],
        [ 0,       sa,       ca,      d],
        [ 0,        0,        0,      1]
    ])

def forward_kinematics_all_frames(dh_params, joint_values):
    """
    Return a list of transformation matrices (one for each frame).
    """
    T = np.eye(4)
    frames = []  # base frame

    for i, (theta, d, a, alpha) in enumerate(dh_params):
        if theta == 'q':  # revolute
            current_theta = joint_values[i]
            current_d = d
        else:  # prismatic
            current_theta = theta
            current_d = joint_values[i]

        A_i = dh_transform(current_theta, current_d, a, alpha)
        T = T @ A_i
        frames.append(T.copy())

    return frames


In [ ]:
def dh_transform_torch(theta, d, a, alpha):
    """Compute DH transformation matrix (torch version, supports autograd)."""
    # ensure everything is a tensor
    theta = torch.as_tensor(theta, dtype=torch.float32, device=d.device if isinstance(d, torch.Tensor) else 'cpu')
    d     = torch.as_tensor(d, dtype=torch.float32, device=theta.device)
    a     = torch.as_tensor(a, dtype=torch.float32, device=theta.device)
    alpha = torch.as_tensor(alpha, dtype=torch.float32, device=theta.device)

    ct, st = torch.cos(theta), torch.sin(theta)
    ca, sa = torch.cos(alpha), torch.sin(alpha)

    T = torch.stack([
        torch.stack([ct, -st * ca,  st * sa, a * ct]),
        torch.stack([st,  ct * ca, -ct * sa, a * st]),
        torch.stack([torch.zeros((), device=theta.device), sa, ca, d]),
        torch.tensor([0., 0., 0., 1.], dtype=torch.float32, device=theta.device)
    ])
    
    return T


def forward_kinematics_all_frames_torch(dh_params, joint_values):
    """
    Compute forward kinematics for all frames (torch version).
    dh_params: list of (theta, d, a, alpha), where theta can be 'q' (revolute) or numeric,
               and d can be 'q' (prismatic) or numeric.
    joint_values: torch tensor of joint values (shape: [n_joints])
    """
    T = torch.eye(4, dtype=joint_values.dtype, device=joint_values.device)
    frames = []

    for i, (theta, d, a, alpha) in enumerate(dh_params):
        if theta == 'q':  # revolute
            current_theta = joint_values[i]
            current_d = torch.as_tensor(d, dtype=joint_values.dtype, device=joint_values.device)
        else:  # prismatic
            current_theta = torch.as_tensor(theta, dtype=joint_values.dtype, device=joint_values.device)
            current_d = joint_values[i]

        A_i = dh_transform_torch(current_theta, current_d, a, alpha)
        T = T @ A_i
        frames.append(T.clone())  # keep a copy

    return frames

In [ ]:
# Example: 6-DOF arm
dh_params = [
    ('q',     0.1273,   0,        np.pi/2),
    ('q',     0.0,     -0.612,    0),
    ('q',     0.0,     -0.5723,   0),
    ('q',     0.163941, 0,        np.pi/2),
    ('q',     0.1157,   0,       -np.pi/2),
    ('q',     0.0922,   0,        0)
]

joint_limits = [
    [-np.pi, np.pi],
    [0.0, -np.pi],
    [-5 * np.pi/12, 5 * np.pi/12],
    [-np.pi, np.pi],
    [-np.pi, np.pi],
    [-np.pi, np.pi]
]

In [ ]:
def square_prism_between_homogeneous(p1, p2, width):
    p1 = np.array(p1, dtype=float)
    p2 = np.array(p2, dtype=float)
    axis = p2 - p1
    length = np.linalg.norm(axis)

    if length == 0:
        raise ValueError("Points must be distinct to define a prism.")

    axis /= length

    # Orthonormal basis perpendicular to axis
    tmp = np.array([1, 0, 0]) if abs(axis[0]) < 0.99 else np.array([0, 1, 0])
    v1 = np.cross(axis, tmp)
    v1 /= np.linalg.norm(v1)

    v2 = np.cross(axis, v1)

    # Scale to half-width
    v1 *= width / 2
    v2 *= width / 2

    # Corners around p1
    corners_p1 = [
        p1 + v1 + v2,
        p1 + v1 - v2,
        p1 - v1 - v2,
        p1 - v1 + v2
    ]

    # Corners around p2
    corners_p2 = [
        p2 + v1 + v2,
        p2 + v1 - v2,
        p2 - v1 - v2,
        p2 - v1 + v2
    ]

    # Stack and convert to homogeneous coordinates
    vertices = np.array(corners_p1 + corners_p2).T  # shape (3, 8)
    homogeneous_vertices = np.vstack([vertices, np.ones((1, 8))])  # shape (4, 8)

    return homogeneous_vertices


In [ ]:
def plot_robot_plotly(dh_params, joint_values, scene):
    # Compute all frame transformations
    transformations = forward_kinematics_all_frames(dh_params, joint_values)

    for i, frame in enumerate(transformations):
        # Use max of |aᵢ| and |dᵢ| to set link depth
        p = transformations[i][0:3, [-1]].flatten()

        if i != 0:
            last_p = transformations[i - 1][0:3, [-1]].flatten()

        else:
            last_p = [0, 0, 0]

        vertices = square_prism_between_homogeneous(p, last_p, 0.05)
        scene.add_solid(vertices, "")
        scene.add_frame(transformation=frame, name=f"Link {i}", axis_size=0.05, color="black")

    return scene

In [ ]:
joint_values = [0, 0, 0, 0, 0, 0]

# Create scene
scene = Viewer3D(title="UR10 Plotly Display", size=3)  # in cm

# Inertial frame
scene.add_frame(transformation=np.eye(4), name="Inertial Reference", axis_size=0.05)

# Plot robot
plot_robot_plotly(dh_params, joint_values, scene)

scene.figure.show(renderer="notebook_connected")

In [ ]:
dataset_size = 10000

# Randomize based on joint limits
'''output_data = []
for low, high in joint_limits:
    random_joint_values = np.random.uniform(
        low=low,
        high=high,
        size=dataset_size,
    )

    output_data.append(random_joint_values)
output_data = np.array(output_data).T'''

# Full randomize
output_data = np.random.uniform(low=0.0, high=np.pi, size=(dataset_size, len(dh_params)))

input_data = []
for j in output_data:
    input_data.append(forward_kinematics_all_frames(dh_params, j)[-1][0:3, [-1]].flatten()[:3])

input_data = np.array(input_data)

In [ ]:
class Model(nn.Module):
  def __init__(self, in_features=3, h1=200, h2=200, h3=200, h4=200, h5=200, h6=200, out_features=len(dh_params)):
    super().__init__()

    # Neural network structure, like literal connections to neurons
    self.fc1 = nn.Linear(in_features, h1)
    self.fc2 = nn.Linear(h1, h2)
    self.fc3 = nn.Linear(h2, h3)
    self.fc4 = nn.Linear(h3, h4)
    self.fc5 = nn.Linear(h4, h5)
    self.fc6 = nn.Linear(h5, h6)
    self.out = nn.Linear(h6, out_features)

  # This is the computation function, put the input, get the output
  def forward(self, x):
    x = F.relu(self.fc1(x))
    x = F.relu(self.fc2(x))
    x = F.relu(self.fc3(x))
    x = F.relu(self.fc4(x))
    x = F.relu(self.fc5(x))
    x = F.relu(self.fc6(x))
    x = self.out(x)

    return x


In [ ]:
torch.manual_seed = 42

model = Model()

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(input_data, output_data, test_size=0.2, random_state=torch.manual_seed)

x_train = torch.FloatTensor(x_train)
x_test = torch.FloatTensor(x_test)
y_train = torch.FloatTensor(y_train)
y_test = torch.FloatTensor(y_test)

In [ ]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
epochs = 100
losses = []

for i in range(epochs):
  # Use model
  y_pred = model.forward(x_train)

  # Decode output
  x_pred = torch.stack([forward_kinematics_all_frames_torch(dh_params, j)[-1][:3, 3] for j in y_pred])

  # Compute loss function
  loss = criterion(x_pred, x_train)

  losses.append(loss.detach().numpy())

  # Only prints every 10 epochs
  if i % 10 == 0:
    print(f"Epoch: {i}, Loss: {loss}")

  optimizer.zero_grad() # Resets the gradient
  loss.backward() # Computes gradient of loss for each neuron
  optimizer.step() # Update weights and biases based on the gradient

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(range(epochs), losses)
plt.ylabel("Error")
plt.xlabel("Epoch")

In [ ]:
# Create scene
scene = Viewer3D(title="UR10 Plotly Display", size=3)  # in cm

joint_values = [0, -np.pi/2, -np.pi/2, np.pi/2, np.pi/2, np.pi/2]

coord = forward_kinematics_all_frames(dh_params, joint_values)[-1][0:3, [-1]].flatten()[:3]
with torch.no_grad(): # Turn off back propagation
    joints_eval = model.forward(torch.FloatTensor(np.array(coord)))

# Add inertial reference
scene.add_frame(transformation=np.eye(4), name="Inertial Reference", axis_size=40)

# Plot robot
scene = plot_robot_plotly(dh_params, np.random.uniform(low=0.0, high=np.pi, size=(dataset_size, 1)), scene)

scene.add_frame(transformation=np.eye(4), name="Inertial Reference", axis_size=40)

# Plot robot
scene = plot_robot_plotly(dh_params, joints_eval, scene)

scene.figure.show(renderer="notebook_connected")